In [1]:
import pandas as pd

# P2P — from URL
p2p = pd.read_csv('https://raw.githubusercontent.com/d-osei/Lending-Club-Loan-Data/refs/heads/master/loans_2007.csv', low_memory=False)

# Bank Loans
bank = pd.read_csv('credit_risk_dataset.csv')

# Microfinance
micro = pd.read_csv('kiva_loans.csv')

# SME
sme = pd.read_csv('sba_loans.csv')

datasets = {
    'P2P Lending': p2p,
    'Bank Loans': bank,
    'Microfinance': micro,
    'SME Lending': sme
}

for name, df in datasets.items():
    print(f"── {name} ──")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()

── P2P Lending ──
Shape: (39252, 41)
Columns: ['loan_amnt', 'int_rate', 'installment', 'emp_length', 'annual_inc', 'loan_status', 'zip_code', 'dti', 'delinq_2yrs', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'last_fico_range_high', 'home_ownership_MORTGAGE', 'home_ownership_NONE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'verification_status_Not Verified', 'verification_status_Source Verified', 'verification_status_Verified', 'purpose_car', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding', 'term_ 36 months', 'term_ 60 months']

── Bank Loans ──
Shape: (32581, 12)
Columns: ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 

In [2]:
print("── Microfinance Repayment ──")
print(micro['repayment_interval'].value_counts())
print(f"\nFunded vs Loan amount diff (proxy for partial funding):")
micro['funding_gap'] = micro['loan_amount'] - micro['funded_amount']
print(micro['funding_gap'].describe())

print("\n── SME Default Distribution ──")
print(sme['Default'].value_counts())
print(f"\nDefault rate: {sme['Default'].mean():.3f}")

print("\n── P2P Loan Status ──")
print(p2p['loan_status'].value_counts())

print("\n── Bank Loan Status ──")
print(bank['loan_status'].value_counts())

── Microfinance Repayment ──
repayment_interval
monthly      342717
irregular    257158
bullet        70728
weekly          602
Name: count, dtype: int64

Funded vs Loan amount diff (proxy for partial funding):
count    671205.000000
mean         56.402046
std         391.901150
min        -400.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       50000.000000
Name: funding_gap, dtype: float64

── SME Default Distribution ──
Default
0    1416
1     686
Name: count, dtype: int64

Default rate: 0.326

── P2P Loan Status ──
loan_status
1    33586
0     5666
Name: count, dtype: int64

── Bank Loan Status ──
loan_status
0    25473
1     7108
Name: count, dtype: int64


In [4]:
import pandas as pd
import numpy as np

# ── Load all 4 ───────────────────────────────────────────
p2p = pd.read_csv('https://raw.githubusercontent.com/d-osei/Lending-Club-Loan-Data/refs/heads/master/loans_2007.csv', low_memory=False)
bank = pd.read_csv('credit_risk_dataset.csv')
micro = pd.read_csv('kiva_loans.csv')
sme = pd.read_csv('sba_loans.csv')

# ══════════════════════════════════════════════════════════
# P2P LENDING
# ══════════════════════════════════════════════════════════
p2p_clean = pd.DataFrame({
    'loan_amount': p2p['loan_amnt'],
    'interest_rate': p2p['int_rate'],
    'term_months': p2p['term_ 36 months'].map({1: 36, 0: 60}),
    'income': p2p['annual_inc'],
    'dti': p2p['dti'],
    'credit_score': p2p['fico_range_high'],
    'employment_length': p2p['emp_length'],
    'home_ownership': p2p[['home_ownership_MORTGAGE','home_ownership_OWN',
                            'home_ownership_RENT','home_ownership_NONE',
                            'home_ownership_OTHER']].idxmax(axis=1).str.replace('home_ownership_',''),
    'default': p2p['loan_status'],
    'lending_medium': 'P2P'
})

# ══════════════════════════════════════════════════════════
# BANK LOANS
# ══════════════════════════════════════════════════════════
bank_clean = pd.DataFrame({
    'loan_amount': bank['loan_amnt'],
    'interest_rate': bank['loan_int_rate'],
    'term_months': np.nan,
    'income': bank['person_income'],
    'dti': bank['loan_percent_income'] * 100,
    'credit_score': np.nan,
    'employment_length': bank['person_emp_length'],
    'home_ownership': bank['person_home_ownership'],
    'default': bank['loan_status'],
    'lending_medium': 'Bank'
})

# ══════════════════════════════════════════════════════════
# MICROFINANCE
# ══════════════════════════════════════════════════════════
micro['funding_gap'] = micro['loan_amount'] - micro['funded_amount']
micro['default'] = ((micro['repayment_interval'] == 'bullet') & 
                    (micro['funding_gap'] > 0)).astype(int)

micro_clean = pd.DataFrame({
    'loan_amount': micro['loan_amount'],
    'interest_rate': np.nan,
    'term_months': micro['term_in_months'],
    'income': np.nan,
    'dti': np.nan,
    'credit_score': np.nan,
    'employment_length': np.nan,
    'home_ownership': 'UNKNOWN',
    'default': micro['default'],
    'lending_medium': 'Microfinance'
})

# ══════════════════════════════════════════════════════════
# SME LENDING — Synthetic Expansion
# ══════════════════════════════════════════════════════════
np.random.seed(42)
n_synthetic = 20000

sme_real = pd.DataFrame({
    'loan_amount': pd.to_numeric(sme['DisbursementGross'], errors='coerce'),
    'interest_rate': np.nan,
    'term_months': sme['Term'],
    'income': np.nan,
    'dti': np.nan,
    'credit_score': np.nan,
    'employment_length': sme['NoEmp'],
    'home_ownership': 'BUSINESS',
    'default': sme['Default'],
    'lending_medium': 'SME'
})

# Synthetic expansion based on real distribution
sme_synthetic = pd.DataFrame({
    'loan_amount': np.random.lognormal(
        mean=np.log(sme_real['loan_amount'].dropna().mean()),
        sigma=0.8, size=n_synthetic
    ),
    'interest_rate': np.random.uniform(4, 12, n_synthetic),
    'term_months': np.random.choice([60, 84, 120, 180, 240], n_synthetic),
    'income': np.random.lognormal(11, 0.7, n_synthetic),
    'dti': np.random.uniform(0.1, 0.5, n_synthetic),
    'credit_score': np.random.normal(680, 60, n_synthetic).clip(300, 850),
    'employment_length': np.random.randint(1, 50, n_synthetic),
    'home_ownership': 'BUSINESS',
    'default': np.random.binomial(1, sme['Default'].mean(), n_synthetic),
    'lending_medium': 'SME'
})

sme_clean = pd.concat([sme_real, sme_synthetic], ignore_index=True)

# ══════════════════════════════════════════════════════════
# COMBINE ALL 4
# ══════════════════════════════════════════════════════════
unified = pd.concat([p2p_clean, bank_clean, micro_clean, sme_clean], 
                     ignore_index=True)

print(f"── Unified Dataset ──")
print(f"Shape: {unified.shape}")
print(f"\nLending Medium Distribution:")
print(unified['lending_medium'].value_counts())
print(f"\nOverall Default Rate: {unified['default'].mean():.3f}")
print(f"\nDefault Rate by Medium:")
print(unified.groupby('lending_medium')['default'].mean().round(3))
print(f"\nNull %:")
print((unified.isnull().sum() / len(unified) * 100).round(1))

── Unified Dataset ──
Shape: (765140, 10)

Lending Medium Distribution:
lending_medium
Microfinance    671205
P2P              39252
Bank             32581
SME              22102
Name: count, dtype: int64

Overall Default Rate: 0.076

Default Rate by Medium:
lending_medium
Bank            0.218
Microfinance    0.015
P2P             0.856
SME             0.334
Name: default, dtype: float64

Null %:
loan_amount           0.0
interest_rate        88.4
term_months           4.3
income               88.0
dti                  88.0
credit_score         92.3
employment_length    87.8
home_ownership        0.0
default               0.0
lending_medium        0.0
dtype: float64


In [5]:
print(p2p['loan_status'].value_counts())


loan_status
1    33586
0     5666
Name: count, dtype: int64


In [6]:
# Invert P2P loan_status — 1 means good, 0 means default
p2p_clean['default'] = (p2p['loan_status'] == 0).astype(int)

print("Fixed P2P default rate:", p2p_clean['default'].mean().round(3))

Fixed P2P default rate: 0.144


In [7]:
unified = pd.concat([p2p_clean, bank_clean, micro_clean, sme_clean], 
                     ignore_index=True)

print("Default Rate by Medium:")
print(unified.groupby('lending_medium')['default'].mean().round(3))
print(f"\nOverall Default Rate: {unified['default'].mean():.3f}")

Default Rate by Medium:
lending_medium
Bank            0.218
Microfinance    0.015
P2P             0.144
SME             0.334
Name: default, dtype: float64

Overall Default Rate: 0.040


In [8]:
# ── Feature Engineering ──────────────────────────────────
np.random.seed(42)
n = len(unified)

# ── Risk Features ────────────────────────────────────────
unified['loan_to_income'] = unified['loan_amount'] / (unified['income'] + 1)
unified['monthly_burden'] = unified['loan_amount'] / (unified['term_months'] + 1)
unified['high_dti_flag'] = (unified['dti'] > 35).astype(int)
unified['long_term_flag'] = (unified['term_months'] > 36).astype(int)
unified['cost_of_credit'] = (unified['interest_rate'] / 100) * unified['term_months']

# ── Loan Size Category ───────────────────────────────────
unified['loan_size'] = pd.cut(
    unified['loan_amount'],
    bins=[0, 5000, 15000, 35000, float('inf')],
    labels=['Micro', 'Small', 'Medium', 'Large']
)

# ── Credit Tier ──────────────────────────────────────────
unified['credit_tier'] = pd.cut(
    unified['credit_score'],
    bins=[0, 580, 670, 740, 800, 850],
    labels=['Poor', 'Fair', 'Good', 'Very Good', 'Exceptional']
)

# ── Income Segment ───────────────────────────────────────
unified['income_segment'] = pd.cut(
    unified['income'],
    bins=[0, 25000, 50000, 100000, float('inf')],
    labels=['Low', 'Middle', 'Upper Middle', 'High']
)

# ── Risk Interaction ─────────────────────────────────────
unified['risk_interaction'] = (
    unified['loan_amount'] * unified['dti'].fillna(unified['dti'].median())
) / (unified['income'].fillna(unified['income'].median()) + 1)

# ── India Emerging Market Context ────────────────────────
unified['digital_onboarding'] = np.random.binomial(1, 0.7, n)
unified['upi_transaction_count'] = np.random.poisson(45, n)
unified['mobile_credit_score'] = np.random.normal(650, 80, n).clip(300, 850)
unified['first_time_borrower'] = np.random.binomial(1, 0.35, n)
unified['urban_flag'] = np.random.binomial(1, 0.65, n)

# ── Risk Tier (Target 2) ─────────────────────────────────
def assign_risk_tier(row):
    prob_proxy = 0
    if row['default'] == 1:
        prob_proxy = np.random.uniform(0.6, 1.0)
    else:
        prob_proxy = np.random.uniform(0.0, 0.4)
    
    if prob_proxy > 0.6:
        return 'High'
    elif prob_proxy > 0.3:
        return 'Medium'
    else:
        return 'Low'

unified['risk_tier'] = unified.apply(assign_risk_tier, axis=1)

# ── Expected Loss (Target 3) ─────────────────────────────
unified['expected_loss'] = unified['loan_amount'] * unified['default'] * np.random.uniform(0.4, 0.9, n)

# ── Impute interest rate by medium median ─────────────────
unified['interest_rate'] = unified.groupby('lending_medium')['interest_rate'].transform(
    lambda x: x.fillna(x.median())
)

print(f"Final Shape: {unified.shape}")
print(f"\nColumns: {unified.columns.tolist()}")
print(f"\nNull %:")
print((unified.isnull().sum() / len(unified) * 100).round(1))
print(f"\nRisk Tier Distribution:")
print(unified['risk_tier'].value_counts())

Final Shape: (765140, 26)

Columns: ['loan_amount', 'interest_rate', 'term_months', 'income', 'dti', 'credit_score', 'employment_length', 'home_ownership', 'default', 'lending_medium', 'loan_to_income', 'monthly_burden', 'high_dti_flag', 'long_term_flag', 'cost_of_credit', 'loan_size', 'credit_tier', 'income_segment', 'risk_interaction', 'digital_onboarding', 'upi_transaction_count', 'mobile_credit_score', 'first_time_borrower', 'urban_flag', 'risk_tier', 'expected_loss']

Null %:
loan_amount               0.0
interest_rate            87.7
term_months               4.3
income                   88.0
dti                      88.0
credit_score             92.3
employment_length        87.8
home_ownership            0.0
default                   0.0
lending_medium            0.0
loan_to_income           88.0
monthly_burden            4.3
high_dti_flag             0.0
long_term_flag            0.0
cost_of_credit           92.3
loan_size                 0.0
credit_tier              92.3
inco

In [9]:
import sqlite3

# ── Smart Null Handling ──────────────────────────────────

# 1. Impute numerics by lending medium median
num_cols = ['interest_rate', 'term_months', 'income', 'dti', 
            'credit_score', 'employment_length', 'loan_to_income',
            'cost_of_credit']

for col in num_cols:
    unified[col] = unified.groupby('lending_medium')[col].transform(
        lambda x: x.fillna(x.median())
    )

# 2. Fill categorical nulls
unified['credit_tier'] = unified['credit_tier'].astype(str).replace('nan', 'Unknown')
unified['income_segment'] = unified['income_segment'].astype(str).replace('nan', 'Unknown')
unified['loan_size'] = unified['loan_size'].astype(str).replace('nan', 'Unknown')

# 3. Encode categoricals
unified['home_ownership_enc'] = unified['home_ownership'].astype('category').cat.codes
unified['lending_medium_enc'] = unified['lending_medium'].astype('category').cat.codes
unified['risk_tier_enc'] = unified['risk_tier'].map({'Low': 0, 'Medium': 1, 'High': 2})
unified['loan_size_enc'] = unified['loan_size'].map({'Micro': 0, 'Small': 1, 'Medium': 2, 'Large': 3}).fillna(0)
unified['credit_tier_enc'] = unified['credit_tier'].map({
    'Poor': 0, 'Fair': 1, 'Good': 2, 'Very Good': 3, 'Exceptional': 4, 'Unknown': -1
})
unified['income_segment_enc'] = unified['income_segment'].map({
    'Low': 0, 'Middle': 1, 'Upper Middle': 2, 'High': 3, 'Unknown': -1
})

print("Null % after imputation:")
print((unified.isnull().sum() / len(unified) * 100).round(1))
print(f"\nFinal shape: {unified.shape}")

Null % after imputation:
loan_amount               0.0
interest_rate            87.7
term_months               4.3
income                   87.7
dti                      87.7
credit_score             92.0
employment_length        87.7
home_ownership            0.0
default                   0.0
lending_medium            0.0
loan_to_income           87.7
monthly_burden            4.3
high_dti_flag             0.0
long_term_flag            0.0
cost_of_credit           92.0
loan_size                 0.0
credit_tier              92.3
income_segment           88.0
risk_interaction          0.0
digital_onboarding        0.0
upi_transaction_count     0.0
mobile_credit_score       0.0
first_time_borrower       0.0
urban_flag                0.0
risk_tier                 0.0
expected_loss             0.0
home_ownership_enc        0.0
lending_medium_enc        0.0
risk_tier_enc             0.0
loan_size_enc             0.0
credit_tier_enc          92.3
income_segment_enc       88.0
dtype: float64


In [10]:
# ── Two-step imputation ──────────────────────────────────
# Step 1: impute by medium median
for col in num_cols:
    unified[col] = unified.groupby('lending_medium')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Step 2: fill remaining nulls with global median
for col in num_cols:
    unified[col] = unified[col].fillna(unified[col].median())

# Fix encoded columns too
unified['credit_tier_enc'] = unified['credit_tier_enc'].fillna(-1)
unified['income_segment_enc'] = unified['income_segment_enc'].fillna(-1)
unified['loan_size_enc'] = unified['loan_size_enc'].fillna(0)
unified['cost_of_credit'] = unified['cost_of_credit'].fillna(
    unified['interest_rate'] / 100 * unified['term_months']
)
unified['loan_to_income'] = unified['loan_to_income'].fillna(
    unified['loan_amount'] / (unified['income'] + 1)
)

print("Null % after fix:")
print((unified.isnull().sum() / len(unified) * 100).round(1))

Null % after fix:
loan_amount               0.0
interest_rate             0.0
term_months               0.0
income                    0.0
dti                       0.0
credit_score              0.0
employment_length         0.0
home_ownership            0.0
default                   0.0
lending_medium            0.0
loan_to_income            0.0
monthly_burden            4.3
high_dti_flag             0.0
long_term_flag            0.0
cost_of_credit            0.0
loan_size                 0.0
credit_tier              92.3
income_segment           88.0
risk_interaction          0.0
digital_onboarding        0.0
upi_transaction_count     0.0
mobile_credit_score       0.0
first_time_borrower       0.0
urban_flag                0.0
risk_tier                 0.0
expected_loss             0.0
home_ownership_enc        0.0
lending_medium_enc        0.0
risk_tier_enc             0.0
loan_size_enc             0.0
credit_tier_enc           0.0
income_segment_enc        0.0
dtype: float64


In [12]:
# Fix monthly_burden
unified['monthly_burden'] = unified['loan_amount'] / (unified['term_months'] + 1)

# Fix credit_tier and income_segment categorical strings
unified['credit_tier'] = unified['credit_tier'].astype(str).replace('nan', 'Unknown').fillna('Unknown')
unified['income_segment'] = unified['income_segment'].astype(str).replace('nan', 'Unknown').fillna('Unknown')

print("Null % final check:")
print((unified.isnull().sum() / len(unified) * 100).round(1))
print("\nAll zeros! ", unified.isnull().sum().sum() == 0)

Null % final check:
loan_amount              0.0
interest_rate            0.0
term_months              0.0
income                   0.0
dti                      0.0
credit_score             0.0
employment_length        0.0
home_ownership           0.0
default                  0.0
lending_medium           0.0
loan_to_income           0.0
monthly_burden           0.0
high_dti_flag            0.0
long_term_flag           0.0
cost_of_credit           0.0
loan_size                0.0
credit_tier              0.0
income_segment           0.0
risk_interaction         0.0
digital_onboarding       0.0
upi_transaction_count    0.0
mobile_credit_score      0.0
first_time_borrower      0.0
urban_flag               0.0
risk_tier                0.0
expected_loss            0.0
home_ownership_enc       0.0
lending_medium_enc       0.0
risk_tier_enc            0.0
loan_size_enc            0.0
credit_tier_enc          0.0
income_segment_enc       0.0
dtype: float64

All zeros!  True


In [13]:
import sqlite3

conn = sqlite3.connect('digital_lending.db')

# ── Table 1: loans (fact table) ──────────────────────────
loans = unified[[
    'loan_amount', 'interest_rate', 'term_months', 'income',
    'dti', 'credit_score', 'employment_length', 'home_ownership',
    'lending_medium', 'default', 'loan_to_income', 'monthly_burden',
    'high_dti_flag', 'long_term_flag', 'cost_of_credit',
    'loan_size', 'credit_tier', 'income_segment', 'risk_interaction',
    'digital_onboarding', 'upi_transaction_count', 'mobile_credit_score',
    'first_time_borrower', 'urban_flag', 'risk_tier', 'expected_loss'
]].copy()
loans['loan_id'] = range(1, len(loans) + 1)
loans.to_sql('loans', conn, if_exists='replace', index=False)

# ── Table 2: medium summary ──────────────────────────────
medium_summary = unified.groupby('lending_medium').agg(
    total_loans=('default', 'count'),
    default_rate=('default', 'mean'),
    avg_loan_amount=('loan_amount', 'mean'),
    avg_interest_rate=('interest_rate', 'mean'),
    avg_term_months=('term_months', 'mean'),
    total_expected_loss=('expected_loss', 'sum')
).round(3).reset_index()
medium_summary.to_sql('medium_summary', conn, if_exists='replace', index=False)

# ── Table 3: risk segments ───────────────────────────────
risk_segments = unified.groupby(['lending_medium', 'risk_tier']).agg(
    loan_count=('default', 'count'),
    default_rate=('default', 'mean'),
    avg_loan_amount=('loan_amount', 'mean'),
    avg_expected_loss=('expected_loss', 'mean')
).round(3).reset_index()
risk_segments.to_sql('risk_segments', conn, if_exists='replace', index=False)

# ── Verify ───────────────────────────────────────────────
for table in ['loans', 'medium_summary', 'risk_segments']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as c FROM {table}", conn)['c'][0]
    print(f"{table}: {count} rows")

conn.close()
print("\nDatabase created!")

loans: 765140 rows
medium_summary: 4 rows
risk_segments: 12 rows

Database created!
